# 02b — LightGBM Baseline · Trained Only on Fresh Lab Readings

**Why this notebook exists.** Notebook 02 trained on *all* rows of the gold table — including the 91.73% that have forward-filled (stale) lab measurements. The result was poor: ~1.5% improvement over the naive baseline, R² negative on `% Iron`. Inspecting the scatter plots showed the model essentially predicting a horizontal line near the train mean.

**The fix.** Restrict training to the 8.27% of rows with **fresh** lab readings — the moments when the lab measurement actually changed (a new sample was processed). This eliminates the forward-fill noise from the supervision signal.

**Hypothesis.** Filtering to fresh-only:
- Test RMSE drops by 30-50%.
- Test R² turns positive (likely 0.4-0.7).
- The scatter plots show genuine spread, not a horizontal stripe.
- Feature importances stay similar (validating the physical interpretation).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from frothiq.data.loader import (
    TARGET_COLS, temporal_split, detect_constant_lab_measurements,
)
from frothiq.features.pipeline import load_gold, list_feature_cols
from frothiq.models.baseline.lightgbm_model import train_one_target

mlflow.set_tracking_uri(f'file:{(ROOT / "mlruns").as_posix()}')
mlflow.set_experiment('frothiq-baseline-fresh')

## 1. Load gold and identify fresh rows

Use `detect_constant_lab_measurements` to flag rows where the lab measurement actually changed.

In [ ]:
gold = load_gold(ROOT / 'data' / 'processed' / 'gold.parquet')
feature_cols = list_feature_cols(gold, target_cols=TARGET_COLS)
print(f'Gold rows: {len(gold):,}')
print(f'Feature cols: {len(feature_cols)}')

# Identify the rows where the lab reading actually changed.
fresh_mask = detect_constant_lab_measurements(gold)
n_fresh = int(fresh_mask.sum())
print(f'\nFresh rows: {n_fresh:,} ({100 * fresh_mask.mean():.2f}%)')
print(f'Forward-filled rows (DROPPED for training): {len(gold) - n_fresh:,}')

In [ ]:
# Keep only fresh rows.
gold_fresh = gold[fresh_mask].reset_index(drop=True)
print(f'Working dataset: {len(gold_fresh):,} fresh rows × {gold_fresh.shape[1]} cols')

## 2. Temporal split on the fresh subset

In [ ]:
train, val, test = temporal_split(gold_fresh, train_frac=0.7, val_frac=0.15)

# Drop any rows with NaN in features (lag features create NaNs at the head of the original gold).
train = train.dropna(subset=feature_cols + TARGET_COLS).reset_index(drop=True)
val = val.dropna(subset=feature_cols + TARGET_COLS).reset_index(drop=True)
test = test.dropna(subset=feature_cols + TARGET_COLS).reset_index(drop=True)

print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')

## 3. Train per-target LightGBM models

In [ ]:
results = {}
for target in TARGET_COLS:
    print(f'\n=== Training {target} ===')
    result = train_one_target(
        X_train=train[feature_cols],
        y_train=train[target].to_numpy(),
        X_val=val[feature_cols],
        y_val=val[target].to_numpy(),
        target_name=target,
        X_test=test[feature_cols],
        y_test=test[target].to_numpy(),
        run_name=f'lgbm-fresh-{target}',
    )
    results[target] = result
    print(f'val: {result.val_metrics}')
    print(f'test: {result.test_metrics}')

## 4. Comparison: naive baseline vs LightGBM (all rows) vs LightGBM (fresh only)

Side-by-side comparison so the value of training on fresh-only is explicit.

In [ ]:
from sklearn.metrics import mean_squared_error

# Pull the previous LightGBM (all-rows) test metrics from MLflow.
prev_runs = mlflow.search_runs(experiment_names=['frothiq-baseline'])
prev_iron = prev_runs[prev_runs['params.target'] == 'pct_iron_concentrate'].iloc[0] if len(prev_runs) else None
prev_silica = prev_runs[prev_runs['params.target'] == 'pct_silica_concentrate'].iloc[0] if len(prev_runs) else None

rows = []
for target in TARGET_COLS:
    train_mean = train[target].mean()
    y_test = test[target].to_numpy()
    naive_pred = np.full_like(y_test, train_mean)
    naive_rmse = float(np.sqrt(mean_squared_error(y_test, naive_pred)))

    lgbm_all_rmse = float(prev_iron['metrics.test_rmse']) if target == 'pct_iron_concentrate' and prev_iron is not None else (
        float(prev_silica['metrics.test_rmse']) if prev_silica is not None else None
    )

    rows.append({
        'target': target,
        'naive_rmse_fresh': naive_rmse,
        'lgbm_all_rows_rmse': lgbm_all_rmse,
        'lgbm_fresh_rmse': results[target].test_metrics['test_rmse'],
        'lgbm_fresh_r2': results[target].test_metrics['test_r2'],
    })
summary = pd.DataFrame(rows)
summary['fresh_vs_all_pct'] = 100 * (1 - summary['lgbm_fresh_rmse'] / summary['lgbm_all_rows_rmse'])
summary['fresh_vs_naive_pct'] = 100 * (1 - summary['lgbm_fresh_rmse'] / summary['naive_rmse_fresh'])
summary.round(3)

## 5. Predicted vs true on test set (fresh)

Compare these scatter plots to the ones in notebook 02 — we expect to see genuine spread along the diagonal, not a horizontal stripe.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, target in zip(axes, TARGET_COLS):
    y_pred = results[target].model.predict(test[feature_cols])
    y_true = test[target].to_numpy()
    ax.scatter(y_true, y_pred, alpha=0.3, s=8)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', alpha=0.6, label='ideal')
    ax.set_xlabel(f'true {target}')
    ax.set_ylabel(f'predicted {target}')
    rmse = results[target].test_metrics['test_rmse']
    r2 = results[target].test_metrics['test_r2']
    ax.set_title(f'{target} — RMSE = {rmse:.3f} · R² = {r2:.3f}')
    ax.legend()
fig.tight_layout()

## 6. Feature importance (fresh model)

Should be physically interpretable: pct_iron_feed important for % Iron Concentrate, starch / amina flow important for % Silica Concentrate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, target in zip(axes, TARGET_COLS):
    model = results[target].model
    imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(20)
    ax.barh(imp.index[::-1], imp.values[::-1])
    ax.set_title(f'{target} — top 20 features (fresh-only model)')
fig.tight_layout()

## 7. Honest reading of the results

Three takeaways for the final report:

1. **Fresh-only training is the methodologically correct path** when 91.73% of labels are forward-filled. Notebook 02 documents the failure mode honestly; this notebook documents the fix.

2. **The improvement vs LightGBM-on-all-rows is the headline number** for the CV. Whatever percent we improve here is the value of doing the right preprocessing.

3. **The fresh-only model is what gets registered to `@production`** in MLflow Model Registry. The all-rows model stays as a counter-example for documentation.

## What's next

- **Notebook 03** (LSTM) — re-train using the same fresh-only filter for fair comparison.
- **Notebook 04** (SPC) — apply control charts to the *residuals* of the fresh-only model.
- **Notebook 05** (What-if) — load the fresh-only model from MLflow and run counterfactual queries.